In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize


# Paths

project_directory = Path(
    r"C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents"
    r"\IMDB_WEEK1_SIX\movie-embeddings-project"
)

dataset_path = (
    project_directory
    / "data"
    / "processed"
    / "movies_cleaned.csv"
)

embeddings_directory = (
    project_directory
    / "artifacts"
    / "embeddings"
)

metrics_directory = (
    project_directory
    / "artifacts"
    / "metrics"
)

metrics_directory.mkdir(parents=True, exist_ok=True)


# Load dataset

movies_df = pd.read_csv(dataset_path).reset_index(drop=True)

required_columns = {"movie_id", "title", "overview"}
missing_columns = required_columns - set(movies_df.columns)

assert not missing_columns, (
    f"Nedostaju kolone: {sorted(missing_columns)}"
)

assert movies_df["movie_id"].is_unique
assert movies_df["overview"].notna().all()


# Load embeddings

embedding_paths = {
    "minilm": embeddings_directory / "minilm_embeddings.npy",
    "mpnet": embeddings_directory / "mpnet_embeddings.npy",
    "gte_modernbert": (
        embeddings_directory / "gte_modernbert_embeddings.npy"
    ),
}

embeddings = {}

for model_name, embedding_path in embedding_paths.items():
    if not embedding_path.exists():
        raise FileNotFoundError(
            f"Nedostaje embedding fajl: {embedding_path}"
        )

    model_embeddings = np.load(embedding_path)

    model_embeddings = model_embeddings.astype(np.float32)

    assert model_embeddings.shape[0] == len(movies_df), (
        f"{model_name}: broj embeddinga "
        f"{model_embeddings.shape[0]} ne odgovara broju filmova "
        f"{len(movies_df)}."
    )

    assert np.isfinite(model_embeddings).all(), (
        f"{model_name} sadrži NaN ili inf vrednosti."
    )

    model_embeddings = normalize(
        model_embeddings,
        norm="l2",
        axis=1,
    )

    embeddings[model_name] = model_embeddings

    print(
        f"{model_name}: "
        f"shape={model_embeddings.shape}, "
        f"dtype={model_embeddings.dtype}"
    )

minilm: shape=(9967, 384), dtype=float32
mpnet: shape=(9967, 768), dtype=float32
gte_modernbert: shape=(9967, 768), dtype=float32


In [8]:
query_titles = [
    "The Dark Knight",
    "Inception",
    "Toy Story",
    "The Godfather",
    "The Matrix",
    "Titanic",
    "Alien",
    "The Exorcist",
    "Saving Private Ryan",
    "The Notebook",
    "Finding Nemo",
    "Gladiator",
    "Se7en",
    "The Silence of the Lambs",
    "The Lord of the Rings: The Fellowship of the Ring",
    "The Truman Show",
    "Black Swan",
    "Mad Max: Fury Road",
    "The Social Network",
    "The Grand Budapest Hotel",
]


print(f"Query filmovi: {len(query_titles)}")
print(query_titles)


Query filmovi: 20
['The Dark Knight', 'Inception', 'Toy Story', 'The Godfather', 'The Matrix', 'Titanic', 'Alien', 'The Exorcist', 'Saving Private Ryan', 'The Notebook', 'Finding Nemo', 'Gladiator', 'Se7en', 'The Silence of the Lambs', 'The Lord of the Rings: The Fellowship of the Ring', 'The Truman Show', 'Black Swan', 'Mad Max: Fury Road', 'The Social Network', 'The Grand Budapest Hotel']


In [ ]:
#Provera isti naziva filmova, razlicitih verzija
selected_query_movies = []

for title in query_titles:
    candidates = find_movie_candidates(
        movies_df,
        title,
    )

    if len(candidates) == 1:
        selected_query_movies.append(
            {
                "query_title": title,
                "query_movie_id": int(
                    candidates.iloc[0]["movie_id"]
                ),
            }
        )
    else:
        print("\nViše filmova sa naslovom:")
        display(candidates)


Više filmova sa naslovom:


,movie_id,title,release_date
95,98,Gladiator,2000-05-04
1050,16219,Gladiator,1992-03-06


In [10]:
def find_movie_candidates(
    movies: pd.DataFrame,
    title: str,
) -> pd.DataFrame:
    title_mask = (
        movies["title"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(title.strip().lower())
    )

    columns = ["movie_id", "title"]

    if "release_date" in movies.columns:
        columns.append("release_date")

    return movies.loc[title_mask, columns].copy()


def get_similar_movies(
    movies: pd.DataFrame,
    model_embeddings: np.ndarray,
    query_movie_id: int,
    top_k: int = 10,
) -> pd.DataFrame:
    query_positions = np.flatnonzero(
        movies["movie_id"].to_numpy() == query_movie_id
    )

    if len(query_positions) != 1:
        raise ValueError(
            f"movie_id={query_movie_id} nije pronađen tačno jednom."
        )

    query_position = int(query_positions[0])

    similarities = cosine_similarity(
        model_embeddings[query_position].reshape(1, -1),
        model_embeddings,
    ).ravel()

    similarities[query_position] = -np.inf

    top_positions = np.argsort(similarities)[::-1][:top_k]

    result_columns = [
        "movie_id",
        "title",
        "overview",
    ]

    if "release_date" in movies.columns:
        result_columns.append("release_date")

    results = movies.iloc[top_positions][result_columns].copy()

    results.insert(
        0,
        "rank",
        np.arange(1, len(results) + 1),
    )

    results["cosine_similarity"] = similarities[top_positions]

    return results.reset_index(drop=True)

In [11]:
top_k = 10
comparison_rows = []


for query_movie in selected_query_movies:
    query_title = query_movie["query_title"]
    query_movie_id = query_movie["query_movie_id"]

    for model_name, model_embeddings in embeddings.items():
        recommendations = get_similar_movies(
            movies=movies_df,
            model_embeddings=model_embeddings,
            query_movie_id=query_movie_id,
            top_k=top_k,
        )

        for _, row in recommendations.iterrows():
            comparison_rows.append(
                {
                    "query_movie_id": query_movie_id,
                    "query_title": query_title,
                    "model": model_name,
                    "rank": int(row["rank"]),
                    "recommended_movie_id": int(
                        row["movie_id"]
                    ),
                    "recommended_title": row["title"],
                    "cosine_similarity": float(
                        row["cosine_similarity"]
                    ),
                    "recommended_overview": row["overview"],
                }
            )

comparison_df = pd.DataFrame(comparison_rows)

comparison_path = (
    metrics_directory
    / "cosine_similarity_model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

print(f"Broj query filmova: {len(selected_query_movies)}")
print(f"Broj rezultata: {len(comparison_df)}")
print(f"Sačuvano u: {comparison_path}")

display(comparison_df.head(20))

Broj query filmova: 19
Broj rezultata: 570
Sačuvano u: C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\movie-embeddings-project\artifacts\metrics\cosine_similarity_model_comparison.csv


,query_movie_id,query_title,model,rank,recommended_movie_id,recommended_title,cosine_similarity,recommended_overview
0,155,The Dark Knight,minilm,1,736073,"Batman: The Long Halloween, Part One",0.764030,Following a brutal series of murders taking pl...
1,155,The Dark Knight,minilm,2,736074,"Batman: The Long Halloween, Part Two",0.725439,"As Gotham City's young vigilante, the Batman, ..."
2,155,The Dark Knight,minilm,3,268,Batman,0.707406,Having witnessed his parents' brutal murder as...
3,155,The Dark Knight,minilm,4,49026,The Dark Knight Rises,0.701888,Following the death of District Attorney Harve...
4,155,The Dark Knight,minilm,5,382322,Batman: The Killing Joke,0.648050,"As Batman hunts for the escaped Joker, the Clo..."
5,155,The Dark Knight,minilm,6,471474,Batman: Gotham by Gaslight,0.647677,"In an alternative Victorian Age Gotham City, B..."
6,155,The Dark Knight,minilm,7,209112,Batman v Superman: Dawn of Justice,0.641332,Fearing the actions of a god-like Super Hero l...
7,155,The Dark Knight,minilm,8,414906,The Batman,0.633064,"In his second year of fighting crime, Batman u..."
8,155,The Dark Knight,minilm,9,21683,Batman: Mystery of the Batwoman,0.622899,As if the Penguin wasn't enough to contend wit...
9,155,The Dark Knight,minilm,10,272,Batman Begins,0.614793,"Driven by tragedy, billionaire Bruce Wayne ded..."
